In [ ]:
# run this notebook after 13_determine_quad_ibd2_filters_R
# use this notebook to merge sibling DNM calls, remove GIAB regions, and apply IBD2 trim
# after this, run 15_prepare_to_calculate_af_R

In [ ]:
library(dplyr)
library(bedtoolsr)
library(data.table)
library(ggplot2)

In [ ]:
sibs_all = fread("./relatedness_dec20/sibs_final.txt")
sibs_all$pair = paste(sibs_all$idv1,sibs_all$idv2, sep = "_")

In [ ]:
# read in ibd2 segment file
ibd2_0cm = fread("./ibd2_all_dec20.txt") 
ibd2_0cm$pair = paste(ibd2_0cm$ID1, ibd2_0cm$ID2, sep = "_")
ibd2_0cm$new_end_bp = ibd2_0cm$stop_coordinate
ibd2_0cm$new_start_bp = ibd2_0cm$start_coordinate
ibd2_0.75cm = fread("./ibd2_all_trimmed_0.75cM_jan12.txt")
ibd2_0.75cm$pair = paste(ibd2_0.75cm$ID1, ibd2_0.75cm$ID2, sep = "_")

In [ ]:
ibd2_all = ibd2_0.75cm %>% filter(length_cm >= 6)
ibd2_all = unique(ibd2_all)

In [ ]:
format_difs = function(difs, ibd_file){
    names(difs) = c("locus", "alleles", "idv1_gt", "idv2_gt")
    duplicated_indices = which(duplicated(difs$locus) | duplicated(difs$locus, fromLast = TRUE))
    difs = difs[-duplicated_indices,]
    difs = difs %>% filter(nchar(alleles) == 9)
    hom00 <- c("0/0","0|0"); het01 <- c("0/1","1/0","0|1","1|0")
    keep <- (difs$idv1_gt %in% hom00 & difs$idv2_gt %in% het01) |
          (difs$idv2_gt %in% hom00 & difs$idv1_gt %in% het01)
    difs <- difs[keep]
    difs$chr = NA
    difs$pos = NA
    chr_pos = gsub(x = difs$locus, pattern = "chr", replace = "")
    difs$chr = gsub(x = chr_pos, pattern = ":.*", replace = "")
    difs$pos = gsub(x = chr_pos, pattern = ".*:", replace = "") 
    difs = difs %>% filter(chr %in% 1:22)
    # keep only the differences in ibd2 
    difs_bed = data.frame(chrom = difs$chr, start = as.numeric(difs$pos), end = as.numeric(difs$pos) + 1)
    merged = bedtoolsr::bt.intersect(a = difs_bed, b = ibd_file, c = TRUE)
    difs$in_to_remove = merged$V4
    difs = difs %>% filter(in_to_remove == 0)
    return(difs)
}

In [ ]:
sibs_all$file_exists = TRUE
for (i in 1:nrow(sibs_all)){
    if (i %% 10 == 0){
        message(i)
    }
    idv1 = sibs_all$idv1[i]
    idv2 = sibs_all$idv2[i]
    f_all_alt = paste("./sibs/difs_all_alt/", idv1, "_", idv2, "_difs_all_alt.tsv", sep = "")
    f_out = paste("./sibs/formatted_difs/", idv1, "_", idv2, "_difs_all_alt_formatted.tsv", sep = "")
    if (file.exists(f_all_alt) & !file.exists(f_out)){
        sibs_all$file_exists[i] = TRUE
        d_all_alt = fread(f_all_alt, sep = "\t", header = TRUE)
        formatted = format_difs(d_all_alt, ibd2_bed)
        fwrite(formatted, f_out, sep = "\t", quote = FALSE, row.names = FALSE)  
    } else {
        sibs_all$file_exists[i] = FALSE
    }
}

In [ ]:
idv1 = sibs_all$idv1[1]
idv2 = sibs_all$idv2[1]
f = paste("./sibs/formatted_difs/", idv1, "_", idv2, "_difs_all_alt_formatted.tsv", sep = "")
all = fread(f, sep = "\t", header = TRUE)
all$idv1 = idv1
all$idv2 = idv2
all = all[,c("chr", "pos", "alleles", "idv1", "idv2","idv1_gt", "idv2_gt")]
sibs_all$num_snps_qc[1] = nrow(all)
for (i in 2:nrow(sibs_all)){
    if (i %% 1000 == 0){
        message(i)
    }
    idv1 = sibs_all$idv1[i]
    idv2 = sibs_all$idv2[i]
    f = paste("./sibs/formatted_difs/", idv1, "_", idv2, "_difs_all_alt_formatted.tsv", sep = "")
    if (file.exists(f)){
        difs = fread(f, sep = "\t", header = TRUE)
        difs$idv1 = idv1
        difs$idv2 = idv2
        difs = difs[,c("chr", "pos", "alleles", "idv1", "idv2","idv1_gt", "idv2_gt")]
        sibs_all$num_snps_qc[i] = nrow(difs)
        all = rbind(all, difs)
    } 
}

In [ ]:
all$ref = substr(all$alleles, 3,3)
all$alt = substr(all$alleles, 7,7)
fwrite(all, "./sibs_snps_QC.txt", sep = "\t", quote = FALSE, row.names = FALSE)
giab = fread("./giab_difficult_merged_oct27.bed")
all_snps_bed = data.frame(chrom = all$chr, start = all$pos, end = all$pos + 1)
snps_filter = bedtoolsr::bt.intersect(a = all_snps_bed, b = giab, c = TRUE)
all$in_giab = snps_filter$V4
snps_filtered = all %>% filter(in_giab == 0)
fwrite(snps_filtered, "./sibs_snps_QC_GIAB.txt", sep = "\t", quote = FALSE, row.names = FALSE) 